# SAMMD canonical lightweight workflow

This notebook exercises the current SAMMD MVP without OpenMM, OpenFF, RDKit, mBuild, or long computations. It validates a YAML template, builds a deterministic lightweight plan, writes `topology.cif`, and demonstrates toy orientation analysis.

Current limitation: `build_system()` returns a planning object, not a full OpenMM simulation system. The current build writes `topology.cif`, `build_summary.json`, and `resolved_config.yaml`; future backend exports reserve `positions.cif`, `interchange.json`, and `system.xml`.

## Create a template configuration

The CLI command `sammd init -o sammd.yaml` writes the same template. Here we write it directly so the notebook stays self-contained.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from sammd.config import CONFIG_TEMPLATE

workspace = TemporaryDirectory()
workdir = Path(workspace.name)
config_path = workdir / "sammd.yaml"
output_dir = workdir / "outputs"
config_path.write_text(CONFIG_TEMPLATE, encoding="utf-8")

print(f"Wrote template config to {config_path}")

## Validate, load, and build a plan

`load_config()` validates YAML through the Pydantic schema. `build_system()` composes the current deterministic planning artifact.

In [ ]:
from sammd import build_system, load_config

config = load_config(config_path)
plan = build_system(config, output_dir=output_dir)

print("Configuration validated")
print(f"Full OpenMM construction available: {plan.full_construction_available}")

## Inspect the plan summary

These values are useful for checking the planned surface, adjusted commensurate dimensions, SAM occupancy, solution counts, and output paths before any heavy backend work is added.

In [ ]:
summary = {
    "surface": f"{plan.slab.metal}({plan.slab.facet})",
    "adjusted_lateral_size_nm": tuple(round(value, 3) for value in plan.slab.lateral_size_nm),
    "binding_site_count": len(plan.binding_sites),
    "sam_placement_count": len(plan.sam_placements.placements),
    "sam_sites_per_side": plan.sam_placements.selected_sites_per_side,
    "solution_counts": plan.solution.molecule_counts,
    "topology_path": str(plan.output_paths.topology),
    "build_summary_path": str(plan.output_paths.build_summary),
    "resolved_config_path": str(plan.output_paths.resolved_config),
    "reserved_positions_path": str(plan.output_paths.positions),
    "reserved_interchange_path": str(plan.output_paths.openff_interchange),
    "reserved_system_path": str(plan.output_paths.openmm_system),
}

for key, value in summary.items():
    print(f"{key}: {value}")

## Write topology.cif

The emitted file is the first structure to inspect in a molecule viewer. It shows the configured surface and SAM anchor placements.

In [ ]:
topology_path = plan.write_topology_cif(overwrite=True)

print(topology_path)
print(f"Exists: {topology_path.exists()}")

## Future direct OpenMM step

When a future SAMMD backend writes `positions.cif`, `interchange.json`, and `system.xml`, use those artifacts from your own OpenMM Python API script for minimization, equilibration, production, and reporters. That step is not runnable in this lightweight release because those backend exports are reserved target artifacts, not current outputs.

## Toy orientation analysis

The analysis helpers operate on plain Python coordinates. This toy cinnamaldehyde-like geometry uses made-up coordinates and a carbonyl oxygen-like target atom to demonstrate the API.

In [ ]:
from sammd.analysis import analyze_orientation

toy_coordinates = [
    (-0.30, 0.00, 0.00),
    (-0.10, 0.15, 0.02),
    (0.10, 0.10, 0.05),
    (0.30, 0.00, 0.12),
    (0.45, -0.08, 0.24),
]
toy_masses = [12.0, 12.0, 12.0, 12.0, 16.0]
orientation = analyze_orientation(
    toy_coordinates,
    atom_index=4,
    masses=toy_masses,
    side="top",
    reactant_label="toy cinnamaldehyde",
)

print(f"COM: {tuple(round(value, 3) for value in orientation.com)}")
print(f"Target point: {orientation.target_point}")
print(f"Orientation vector: {tuple(round(value, 3) for value in orientation.vector)}")
print(f"Angle to top normal: {orientation.angle_degrees:.2f} degrees")

## Cleanup

The temporary directory can be removed when you are done inspecting generated files. Comment this out if you want to keep `topology.cif`.

In [ ]:
workspace.cleanup()